In [1]:
import numpy as np
import sys , os
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.path.join(os.getcwd(),'..','src'))

from decision_trees import DecisionTree
from pruning import cost_complexity_pruning, _prune_tree, _count_leaves
from utils import accuracy

In [2]:
X_train = np.load('../Data/processed/bc_X_train.npy')
X_test = np.load('../Data/processed/bc_X_test.npy')
y_train = np.load('../Data/processed/bc_y_train.npy')
y_test = np.load('../Data/processed/bc_y_test.npy') 

In [3]:
full_tree = DecisionTree(
    criterion='gini',
    max_depth=4,
    min_samples_split=2,
    min_impurity_decrease=0.0,
    task='classification'
)
full_tree.fit(X_train, y_train)

full_train_acc = accuracy(y_train, full_tree.predict(X_train))
full_test_acc = accuracy(y_test, full_tree.predict(X_test))
full_leaves = _count_leaves(full_tree.root)

print('Unpruned baseline:')
print('train accuracy:', full_train_acc)
print('test accuracy :', full_test_acc)
print('leaves        :', full_leaves) 

Unpruned baseline:
train accuracy: 0.9956043956043956
test accuracy : 0.9385964912280702
leaves        : 12


In [5]:
path = cost_complexity_pruning(full_tree.root, X_train, y_train, task='classification')
print('Trees in pruning path:', len(path))

KeyboardInterrupt: 

In [4]:
import copy

path = cost_complexity_pruning(full_tree.root, X_train, y_train, task='classification')
print('Trees in pruning path:', len(path))

path_rows = []
for alpha, tree_root in path:
    preds = np.array([full_tree._predict_single(row, tree_root) for row in X_test])
    te_acc = accuracy(y_test, preds)
    leaves = _count_leaves(tree_root)
    path_rows.append((alpha, te_acc, leaves))
    print(f'alpha={alpha:.6f}, test_acc={te_acc:.6f}, leaves={leaves}')

KeyboardInterrupt: 

In [ ]:
if path_rows:
    alphas = [r[0] for r in path_rows]
    accs = [r[1] for r in path_rows]

    plt.figure(figsize=(7, 4))
    plt.plot(alphas, accs, marker='o')
    plt.xlabel('alpha')
    plt.ylabel('Test accuracy')
    plt.title('Cost-Complexity Pruning Path')
    plt.tight_layout()
    plt.savefig('../results/pruning_path.png', dpi=160)
    plt.close() 

In [ ]:
if path_rows:
    alphas = [r[0] for r in path_rows]
    accs = [r[1] for r in path_rows]

    plt.figure(figsize=(7, 4))
    plt.plot(alphas, accs, marker='o')
    plt.xlabel('alpha')
    plt.ylabel('Test accuracy')
    plt.title('Cost-Complexity Pruning Path')
    plt.tight_layout()
    plt.show()

In [ ]:
if path_rows:
    best_alpha, best_test_acc, _ = max(path_rows, key=lambda t: t[1])
else:
    best_alpha, best_test_acc = 0.0, full_test_acc

pruned_root = _prune_tree(full_tree.root, best_alpha, X_train, y_train, task='classification')
pruned_tree = full_tree
pruned_tree.root = pruned_root

pruned_train_acc = accuracy(y_train, pruned_tree.predict(X_train))
pruned_test_acc = accuracy(y_test, pruned_tree.predict(X_test))
pruned_leaves = _count_leaves(pruned_root)

print('\nBest pruned tree:')
print('best alpha    :', best_alpha)
print('train accuracy :', pruned_train_acc)
print('test accuracy  :', pruned_test_acc)
print('leaves         :', pruned_leaves) 

In [ ]:
print('\nComparison:')
print('Unpruned tree — train accuracy:', full_train_acc, ', test accuracy:', full_test_acc, ', leaves:', full_leaves)
print('Pruned tree   — train accuracy:', pruned_train_acc, ', test accuracy:', pruned_test_acc, ', leaves:', pruned_leaves)

In [ ]:
print('\nMarkdown summary:')
print(f'Pruning removed {full_leaves - pruned_leaves} leaves and changed test accuracy from {full_test_acc:.4f} to {pruned_test_acc:.4f}.')
print('If test accuracy improved, pruning traded a small amount of training fit for better generalization.') 

In [6]:
# grow tree
full_tree = DecisionTree(criterion='gini', max_depth=5, task='classification')
full_tree.fit(X_train, y_train)

# manually prune with fixed alpha
pruned_root = _prune_tree(full_tree.root, 0.01, X_train, y_train, task='classification')
full_tree.root = pruned_root

# evaluate
print('Pruned test acc:', accuracy(y_test, full_tree.predict(X_test)))
print('Leaves:', _count_leaves(full_tree.root))

Pruned test acc: 0.956140350877193
Leaves: 9
